# DOTS-MOCR Examples

This notebook reproduces the upstream `rednote-hilab/dots.mocr` README tasks with `mlx-vlm` only.
It uses the bundled demo assets under `examples/images`, runs the unique scenarios, and saves raw outputs,
overlays, rendered SVG previews, and a contact sheet. If the model returns malformed SVG, the notebook falls back to a
readable text overlay instead of trying to render an error page.

Covered tasks:
- document parsing
- web parsing
- scene spotting
- image to SVG
- general image QA
- parser-style image, PDF, layout-only, and OCR runs


## Requirements

- `mlx-vlm`
- `Pillow`
- macOS `qlmanage` for SVG rendering previews

The notebook writes artifacts to `~/.cache/mlx_vlm/dots_mocr_examples` by default so it does not dirty the repo.


In [ ]:
from __future__ import annotations

import json
import math
import re
import shutil
import subprocess
from pathlib import Path
from xml.etree import ElementTree as ET

from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFont, ImageOps

from mlx_vlm import apply_chat_template, generate, load


In [ ]:
MODEL = "rednote-hilab/dots.mocr"
MAX_TOKENS = 10_000
OUTPUT_ROOT = Path.home() / ".cache" / "mlx_vlm" / "dots_mocr_examples"
PROJECT_ROOT = Path.cwd()
ASSET_ROOT = PROJECT_ROOT / "examples" / "images"
if not ASSET_ROOT.exists():
    ASSET_ROOT = PROJECT_ROOT / "images"
if not ASSET_ROOT.exists():
    raise FileNotFoundError("Expected bundled assets under examples/images")
OUTPUTS_DIR = OUTPUT_ROOT / "outputs"
OVERLAYS_DIR = OUTPUT_ROOT / "overlays"
RENDERED_DIR = OUTPUT_ROOT / "rendered"

for directory in (OUTPUTS_DIR, OVERLAYS_DIR, RENDERED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

PROMPTS = {
    "layout_all": """Please output the layout information from the PDF image, including each layout element's bbox, its category, and the corresponding text content within the bbox.

1. Bbox format: [x1, y1, x2, y2]

2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title'].

3. Text Extraction & Formatting Rules:
    - Picture: For the 'Picture' category, the text field should be omitted.
    - Formula: Format its text as LaTeX.
    - Table: Format its text as HTML.
    - All Others (Text, Title, etc.): Format their text as Markdown.

4. Constraints:
    - The output text must be the original text from the image, with no translation.
    - All layout elements must be sorted according to human reading order.

5. Final Output: The entire output must be a single JSON object.""",
    "layout_only": "Please output the layout information from this PDF image, including each layout's bbox and its category. The bbox should be in the format [x1, y1, x2, y2]. The layout categories for the PDF document include ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title']. Do not output the corresponding text. The layout result should be in JSON format.",
    "ocr": "Extract the text content from this image.",
    "web": "Parsing the layout info of this webpage image with format json:\n",
    "scene": "Detect and recognize the text in the image.",
    "svg": "Please generate the SVG code based on the image.viewBox=\"0 0 {width} {height}\"",
    "general": "You are a helpful assistant.\n\nPlease describe the content of this image.",
}

ALIASES = {
    "demo_hf_layout": "demo_document_parsing",
    "parser_image_default": "demo_document_parsing",
}


##  Utils

In [ ]:
COLORS = {
    "Text": (0, 128, 0),
    "Picture": (255, 0, 255),
    "Caption": (255, 165, 0),
    "Section-header": (0, 255, 255),
    "Footnote": (0, 128, 0),
    "Formula": (128, 128, 128),
    "Table": (255, 105, 180),
    "Title": (255, 0, 0),
    "List-item": (0, 0, 255),
    "Page-header": (34, 139, 34),
    "Page-footer": (128, 0, 128),
    "Other": (165, 42, 42),
    "Unknown": (90, 90, 90),
}

FONT_CANDIDATES = [
    "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
    "/System/Library/Fonts/Helvetica.ttc",
    "/System/Library/Fonts/Supplemental/Arial.ttf",
]

def load_font(size: int):
    for candidate in FONT_CANDIDATES:
        path = Path(candidate)
        if path.exists():
            try:
                return ImageFont.truetype(str(path), size=size)
            except Exception:
                pass
    return ImageFont.load_default()

def text_bbox(draw: ImageDraw.ImageDraw, text: str, font):
    box = draw.textbbox((0, 0), text, font=font)
    return box[2] - box[0], box[3] - box[1]

def round_by_factor(number: int, factor: int) -> int:
    return round(number / factor) * factor

def ceil_by_factor(number: int, factor: int) -> int:
    return math.ceil(number / factor) * factor

def floor_by_factor(number: int, factor: int) -> int:
    return math.floor(number / factor) * factor

def smart_resize(height: int, width: int, factor: int = 28, min_pixels: int = 3136, max_pixels: int = 11289600):
    if max(height, width) / min(height, width) > 200:
        raise ValueError("absolute aspect ratio must be smaller than 200")

    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = max(factor, floor_by_factor(height / beta, factor))
        w_bar = max(factor, floor_by_factor(width / beta, factor))
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
        if h_bar * w_bar > max_pixels:
            beta = math.sqrt((h_bar * w_bar) / max_pixels)
            h_bar = max(factor, floor_by_factor(h_bar / beta, factor))
            w_bar = max(factor, floor_by_factor(w_bar / beta, factor))
    return h_bar, w_bar

def rescale_layout_cells(cells, image_size, min_pixels: int = 3136, max_pixels: int = 11289600):
    original_width, original_height = image_size
    input_height, input_width = smart_resize(
        original_height,
        original_width,
        min_pixels=min_pixels,
        max_pixels=max_pixels,
    )
    scale_x = input_width / original_width
    scale_y = input_height / original_height
    scaled = []
    for cell in cells or []:
        bbox = cell.get("bbox")
        if not isinstance(bbox, list) or len(bbox) != 4:
            scaled.append(cell)
            continue
        cell_copy = dict(cell)
        cell_copy["bbox"] = [
            int(float(bbox[0]) / scale_x),
            int(float(bbox[1]) / scale_y),
            int(float(bbox[2]) / scale_x),
            int(float(bbox[3]) / scale_y),
        ]
        scaled.append(cell_copy)
    return scaled

def rescale_scene_instances(instances, image_size, min_pixels: int = 3136, max_pixels: int = 11289600):
    original_width, original_height = image_size
    input_height, input_width = smart_resize(
        original_height,
        original_width,
        min_pixels=min_pixels,
        max_pixels=max_pixels,
    )
    scale_x = input_width / original_width
    scale_y = input_height / original_height
    scaled = []
    for inst in instances or []:
        points = inst["points"]
        new_points = [
            int(points[i] / (scale_x if i % 2 == 0 else scale_y))
            for i in range(len(points))
        ]
        scaled.append({"points": new_points, "text": inst["text"]})
    return scaled

def parse_partial_json_list(raw: str):
    start = raw.find("[")
    if start == -1:
        return None
    decoder = json.JSONDecoder()
    pos = start + 1
    items = []
    while pos < len(raw):
        while pos < len(raw) and raw[pos] in " \n\r\t,":
            pos += 1
        if pos >= len(raw) or raw[pos] == "]":
            break
        try:
            obj, end = decoder.raw_decode(raw, pos)
        except json.JSONDecodeError:
            break
        if isinstance(obj, dict):
            items.append(obj)
        pos = end
    return items or None

def extract_balanced_json(raw: str):
    starts = [i for i in (raw.find("["), raw.find("{")) if i != -1]
    if not starts:
        return None
    start = min(starts)
    opening = raw[start]
    closing = "]" if opening == "[" else "}"
    depth = 0
    in_str = False
    esc = False
    for idx in range(start, len(raw)):
        ch = raw[idx]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == opening:
            depth += 1
        elif ch == closing:
            depth -= 1
            if depth == 0:
                return raw[start:idx + 1]
    return None

def parse_layout_output(raw: str):
    raw = raw.strip()
    for candidate in (raw, extract_balanced_json(raw)):
        if not candidate:
            continue
        try:
            data = json.loads(candidate)
        except Exception:
            continue
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            for key in ("cells", "layout"):
                value = data.get(key)
                if isinstance(value, list):
                    return value
    return parse_partial_json_list(raw)

def parse_scene_output(raw: str):
    pattern = re.compile(
        r"<\|box_start\|>\((\d+),\s*(\d+)\),\s*"
        r"\((\d+),\s*(\d+)\),\s*"
        r"\((\d+),\s*(\d+)\),\s*"
        r"\((\d+),\s*(\d+)\)<\|box_end\|><\|ref_start\|>(.*?)<\|ref_end\|>",
        re.DOTALL,
    )
    items = []
    for match in pattern.finditer(raw):
        pts = [int(match.group(i)) for i in range(1, 9)]
        items.append({"points": pts, "text": match.group(9).strip()})
    return items

def fix_svg(svg: str):
    if re.search(r'(<path\b[^>]*\bd="[^">]*$)', svg):
        svg += '">'
    svg = re.sub(r"<[^>]*$", "", svg)
    tag_re = re.compile(r"<(/?)([A-Za-z_:][\w:.-]*)([^<>]*?)(/?)>")
    stack = []
    for match in tag_re.finditer(svg):
        closing, name, _, self_close = match.groups()
        name = name.lower()
        if closing:
            if stack and stack[-1] == name:
                stack.pop()
            elif name in stack:
                while stack and stack[-1] != name:
                    stack.pop()
                if stack and stack[-1] == name:
                    stack.pop()
        elif not self_close and name not in {"path", "rect", "circle", "ellipse", "line", "polyline", "polygon", "stop", "use"}:
            stack.append(name)
    while stack:
        svg += f"</{stack.pop()}>"
    return svg

def extract_svg(raw: str):
    stripped = raw.replace("svg:", "").strip()
    full = re.search(r"<svg[^>]*>.*?</svg>", stripped, re.DOTALL)
    if full:
        return full.group(0)
    partial = re.search(r"<svg[^>]*>.*", stripped, re.DOTALL)
    if partial:
        return fix_svg(partial.group(0))
    return None


In [ ]:
def wrap_text(draw, text: str, font, max_width: int, max_lines: int):
    lines = []
    for para in text.splitlines() or [""]:
        words = para.split(" ")
        current = ""
        for word in words:
            candidate = word if not current else current + " " + word
            width, _ = text_bbox(draw, candidate, font)
            if width <= max_width:
                current = candidate
                continue
            if current:
                lines.append(current)
                current = word
            else:
                chunk = ""
                for ch in word:
                    candidate = chunk + ch
                    width, _ = text_bbox(draw, candidate, font)
                    if width <= max_width:
                        chunk = candidate
                    else:
                        if chunk:
                            lines.append(chunk)
                        chunk = ch
                        if len(lines) >= max_lines:
                            return lines[:max_lines], True
                current = chunk
            if len(lines) >= max_lines:
                return lines[:max_lines], True
        if current:
            lines.append(current)
        if len(lines) >= max_lines:
            return lines[:max_lines], True
    return lines[:max_lines], len(lines) > max_lines

def save_layout_overlay(image_path: Path, cells, out_path: Path, title: str):
    base = Image.open(image_path).convert("RGBA")
    overlay = Image.new("RGBA", base.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    label_font = load_font(max(16, min(24, max(14, min(base.size) // 70))))
    banner_font = load_font(max(18, min(28, base.width // 60)))
    line_width = max(2, base.width // 900)

    for idx, cell in enumerate(cells or []):
        bbox = cell.get("bbox")
        if not isinstance(bbox, list) or len(bbox) != 4:
            continue
        x0, y0, x1, y1 = [int(round(v)) for v in bbox]
        category = str(cell.get("category", "Unknown"))
        color = COLORS.get(category, COLORS["Other"])
        draw.rectangle(
            (x0, y0, x1, y1),
            fill=(*color, 58),
            outline=(*color, 230),
            width=line_width,
        )
        snippet = ""
        text = str(cell.get("text", "")).strip()
        if text and category not in {"Table", "Picture"}:
            snippet = " " + text.replace("\n", " ")[:24]
        label = f"{idx}:{category}{snippet}"
        tw, th = text_bbox(draw, label, label_font)
        lx = max(0, min(x0, base.width - tw - 8))
        ly = max(0, y0 - th - 6)
        draw.rounded_rectangle(
            (lx, ly, lx + tw + 8, ly + th + 6),
            radius=4,
            fill=(255, 255, 255, 180),
        )
        draw.text((lx + 4, ly + 3), label, font=label_font, fill=(0, 0, 0, 255))

    bw, bh = text_bbox(draw, title, banner_font)
    draw.rounded_rectangle(
        (12, 12, 24 + bw, 24 + bh),
        radius=8,
        fill=(0, 0, 0, 170),
    )
    draw.text((18, 17), title, font=banner_font, fill=(255, 255, 255, 255))
    Image.alpha_composite(base, overlay).convert("RGB").save(out_path)

def save_scene_overlay(image_path: Path, instances, out_path: Path, title: str):
    base = Image.open(image_path).convert("RGBA")
    overlay = Image.new("RGBA", base.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    label_font = load_font(max(16, min(24, max(14, min(base.size) // 75))))
    banner_font = load_font(max(18, min(28, base.width // 60)))
    line_width = max(2, base.width // 700)

    for idx, inst in enumerate(instances or []):
        points = inst["points"]
        poly = [(points[i], points[i + 1]) for i in range(0, 8, 2)]
        draw.polygon(poly, fill=(0, 255, 0, 36), outline=(0, 255, 0, 230))
        draw.line(poly + [poly[0]], fill=(0, 255, 0, 255), width=line_width)
        label = f"{idx}:{inst['text'][:32]}"
        tw, th = text_bbox(draw, label, label_font)
        x, y = poly[0]
        ly = max(0, y - th - 8)
        draw.rounded_rectangle(
            (x, ly, min(base.width, x + tw + 8), ly + th + 6),
            radius=4,
            fill=(0, 0, 0, 180),
        )
        draw.text((x + 4, ly + 3), label, font=label_font, fill=(255, 255, 255, 255))

    bw, bh = text_bbox(draw, title, banner_font)
    draw.rounded_rectangle(
        (12, 12, 24 + bw, 24 + bh),
        radius=8,
        fill=(0, 0, 0, 170),
    )
    draw.text((18, 17), title, font=banner_font, fill=(255, 255, 255, 255))
    Image.alpha_composite(base, overlay).convert("RGB").save(out_path)

def save_text_overlay(image_path: Path, text: str, out_path: Path, title: str):
    base = Image.open(image_path).convert("RGBA")
    overlay = Image.new("RGBA", base.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    title_font = load_font(max(18, min(28, base.width // 55)))
    text_font = load_font(max(15, min(22, base.width // 65)))
    margin = max(18, base.width // 80)
    panel_h = max(int(base.height * 0.35), 280)
    y0 = max(0, base.height - panel_h)
    draw.rectangle((0, y0, base.width, base.height), fill=(0, 0, 0, 170))
    draw.text((margin, y0 + margin), title, font=title_font, fill=(255, 255, 255, 255))
    _, th = text_bbox(draw, "Ag", text_font)
    max_lines = max(3, (base.height - y0 - (margin * 3 + th)) // (th + 6))
    lines, truncated = wrap_text(
        draw,
        text.strip().replace("\t", "    "),
        text_font,
        base.width - 2 * margin,
        max_lines,
    )
    y = y0 + margin + th + 18
    for line in lines:
        draw.text((margin, y), line, font=text_font, fill=(245, 245, 245, 255))
        y += th + 6
    if truncated:
        draw.text(
            (margin, min(base.height - margin - th, y)),
            "[preview clipped, see .txt output]",
            font=text_font,
            fill=(255, 220, 120, 255),
        )
    Image.alpha_composite(base, overlay).convert("RGB").save(out_path)

def render_preview(input_path: Path, output_dir: Path, size: int):
    if shutil.which("qlmanage") is None:
        raise RuntimeError("qlmanage is required on macOS to render PDF and SVG previews")
    output_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["qlmanage", "-t", "-s", str(size), "-o", str(output_dir), str(input_path)],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    preview_path = output_dir / f"{input_path.name}.png"
    if not preview_path.exists():
        raise FileNotFoundError(f"Quick Look did not render {input_path}")
    return preview_path

def save_svg_overlay(image_path: Path, raw_text: str, overlay_path: Path, svg_path: Path, render_path: Path, comparison_path: Path, title: str):
    svg = extract_svg(raw_text)
    if not svg:
        save_text_overlay(image_path, raw_text, overlay_path, title + " (no SVG extracted)")
        return {"svg_saved": False, "render_saved": False}

    try:
        ET.fromstring(svg)
    except ET.ParseError as error:
        save_text_overlay(
            image_path,
            f"SVG render skipped because the model output is not valid SVG.\n\n{error}\n\n{raw_text}",
            overlay_path,
            title + " (invalid SVG output)",
        )
        return {"svg_saved": False, "render_saved": False, "svg_error": str(error)}

    svg_path.write_text(svg, encoding="utf-8")
    try:
        quicklook_preview = render_preview(svg_path, render_path.parent, size=1200)
    except Exception as error:
        save_text_overlay(
            image_path,
            f"SVG render failed after extraction.\n\n{error}\n\n{raw_text}",
            overlay_path,
            title + " (SVG render failed)",
        )
        return {"svg_saved": True, "render_saved": False, "svg_error": str(error)}
    if quicklook_preview != render_path:
        shutil.move(str(quicklook_preview), str(render_path))

    original = Image.open(image_path).convert("RGBA")
    rendered = Image.open(render_path).convert("RGBA").resize(original.size)
    blended = Image.blend(original, rendered, 0.5)
    draw = ImageDraw.Draw(blended)
    font = load_font(max(18, min(28, original.width // 55)))
    bw, bh = text_bbox(draw, title, font)
    draw.rounded_rectangle(
        (12, 12, 24 + bw, 24 + bh),
        radius=8,
        fill=(0, 0, 0, 170),
    )
    draw.text((18, 17), title, font=font, fill=(255, 255, 255, 255))
    blended.convert("RGB").save(overlay_path)

    comparison = Image.new("RGB", (original.width * 2, original.height), (255, 255, 255))
    comparison.paste(original.convert("RGB"), (0, 0))
    comparison.paste(rendered.convert("RGB"), (original.width, 0))
    cdraw = ImageDraw.Draw(comparison)
    cfont = load_font(max(16, min(24, original.width // 65)))
    cdraw.rectangle((0, 0, original.width, 36), fill=(0, 0, 0))
    cdraw.rectangle((original.width, 0, original.width * 2, 36), fill=(0, 0, 0))
    cdraw.text((12, 8), "original", font=cfont, fill=(255, 255, 255))
    cdraw.text((original.width + 12, 8), "generated SVG render", font=cfont, fill=(255, 255, 255))
    comparison.save(comparison_path)
    return {"svg_saved": True, "render_saved": True}

def make_contact_sheet(results, path: Path):
    font = load_font(24)
    small = load_font(18)
    thumb_w, thumb_h = 820, 460
    margin = 24
    label_h = 70
    cols = 2
    rows = (len(results) + cols - 1) // cols
    canvas = Image.new(
        "RGB",
        (
            cols * (thumb_w + margin) + margin,
            rows * (thumb_h + label_h + margin) + margin,
        ),
        (245, 245, 245),
    )
    draw = ImageDraw.Draw(canvas)
    for idx, item in enumerate(results):
        img = Image.open(item["overlay"]).convert("RGB")
        thumb = ImageOps.contain(img, (thumb_w, thumb_h))
        x = margin + (idx % cols) * (thumb_w + margin)
        y = margin + (idx // cols) * (thumb_h + label_h + margin)
        frame = Image.new("RGB", (thumb_w, thumb_h), (255, 255, 255))
        frame.paste(thumb, ((thumb_w - thumb.width) // 2, (thumb_h - thumb.height) // 2))
        canvas.paste(frame, (x, y))
        draw.rectangle((x, y, x + thumb_w, y + thumb_h), outline=(200, 200, 200), width=2)
        draw.text((x, y + thumb_h + 8), item["id"], font=font, fill=(25, 25, 25))
        draw.text(
            (x, y + thumb_h + 40),
            f"tokens={item['generation_tokens']} capped={item['capped']}",
            font=small,
            fill=(80, 80, 80),
        )
    canvas.save(path)


## Load the model

This loads the processor once and reuses it across all scenarios.


In [ ]:
model, processor = load(MODEL, trust_remote_code=True)
model


## Scenario setup

The upstream README has two duplicate image examples:
- `demo_hf_layout` is the same prompt and image as `demo_document_parsing`
- `parser_image_default` is the same prompt and image as `demo_document_parsing`

This notebook runs the unique scenarios once, then writes alias artifacts for those duplicate entries.


In [ ]:
doc_image = ASSET_ROOT / "demo_image1.jpg"
svg_image = ASSET_ROOT / "demo_image2.png"
general_image = ASSET_ROOT / "demo_image3.jpg"
pdf_file = ASSET_ROOT / "demo_pdf1.pdf"
web_image = ASSET_ROOT / "webpage_1.png"
scene_image = ASSET_ROOT / "scene_1.jpg"
pdf_image = ASSET_ROOT / "demo_pdf1_page1.png"

SCENARIOS = [
    {
        "id": "demo_document_parsing",
        "image": doc_image,
        "prompt": PROMPTS["layout_all"],
        "kind": "layout",
        "temperature": 0.1,
        "top_p": 0.9,
        "title": "document parsing overlay",
    },
    {
        "id": "demo_web_parsing",
        "image": web_image,
        "prompt": PROMPTS["web"],
        "kind": "layout",
        "temperature": 0.1,
        "top_p": 0.9,
        "title": "web parsing overlay",
    },
    {
        "id": "demo_scene_spotting",
        "image": scene_image,
        "prompt": PROMPTS["scene"],
        "kind": "scene",
        "temperature": 0.1,
        "top_p": 0.9,
        "title": "scene spotting overlay",
    },
    {
        "id": "demo_svg",
        "image": svg_image,
        "prompt": PROMPTS["svg"],
        "kind": "svg",
        "temperature": 0.9,
        "top_p": 1.0,
        "title": "svg overlay 50/50 original + generated",
    },
    {
        "id": "demo_general",
        "image": general_image,
        "prompt": PROMPTS["general"],
        "kind": "text",
        "temperature": 0.1,
        "top_p": 0.9,
        "title": "general QA response overlay",
    },
    {
        "id": "parser_pdf_default",
        "image": pdf_image,
        "prompt": PROMPTS["layout_all"],
        "kind": "layout",
        "temperature": 0.1,
        "top_p": 0.9,
        "title": "pdf parser overlay",
    },
    {
        "id": "parser_layout_only",
        "image": doc_image,
        "prompt": PROMPTS["layout_only"],
        "kind": "layout",
        "temperature": 0.1,
        "top_p": 0.9,
        "title": "layout-only overlay",
    },
    {
        "id": "parser_ocr",
        "image": doc_image,
        "prompt": PROMPTS["ocr"],
        "kind": "text",
        "temperature": 0.1,
        "top_p": 0.9,
        "title": "OCR response overlay",
    },
]

SCENARIO_INDEX = {scenario["id"]: scenario for scenario in SCENARIOS}
SCENARIO_ORDER = [
    "demo_document_parsing",
    "demo_web_parsing",
    "demo_scene_spotting",
    "demo_svg",
    "demo_general",
    "demo_hf_layout",
    "parser_image_default",
    "parser_pdf_default",
    "parser_layout_only",
    "parser_ocr",
]

SCENARIOS


In [ ]:
def run_scenario(scenario):
    image_path = Path(scenario["image"])
    raw_path = OUTPUTS_DIR / f"{scenario['id']}.txt"
    overlay_path = OVERLAYS_DIR / f"{scenario['id']}.png"
    prompt = scenario["prompt"]
    if scenario["kind"] == "svg":
        with Image.open(image_path) as img:
            prompt = prompt.format(width=img.width, height=img.height)

    formatted_prompt = apply_chat_template(
        processor,
        model.config,
        prompt,
        num_images=1,
        add_generation_prompt=True,
    )
    result = generate(
        model,
        processor,
        formatted_prompt,
        image=str(image_path),
        max_tokens=MAX_TOKENS,
        temperature=scenario["temperature"],
        top_p=scenario["top_p"],
        verbose=False,
    )
    raw_path.write_text(result.text, encoding="utf-8")

    parse_ok = None
    extra = {}
    if scenario["kind"] == "layout":
        cells = parse_layout_output(result.text)
        parse_ok = cells is not None
        if cells is None:
            save_text_overlay(
                image_path,
                result.text,
                overlay_path,
                scenario["title"] + " (parse failed)",
            )
        else:
            with Image.open(image_path) as img:
                cells = rescale_layout_cells(cells, img.size)
            save_layout_overlay(image_path, cells, overlay_path, scenario["title"])
            extra["cell_count"] = len(cells)
    elif scenario["kind"] == "scene":
        instances = parse_scene_output(result.text)
        parse_ok = bool(instances)
        if instances:
            with Image.open(image_path) as img:
                instances = rescale_scene_instances(instances, img.size)
            save_scene_overlay(image_path, instances, overlay_path, scenario["title"])
            extra["instance_count"] = len(instances)
        else:
            save_text_overlay(
                image_path,
                result.text,
                overlay_path,
                scenario["title"] + " (parse failed)",
            )
    elif scenario["kind"] == "svg":
        svg_path = RENDERED_DIR / f"{scenario['id']}.svg"
        render_path = RENDERED_DIR / f"{scenario['id']}.png"
        comparison_path = RENDERED_DIR / f"{scenario['id']}_comparison.png"
        extra.update(
            save_svg_overlay(
                image_path,
                result.text,
                overlay_path,
                svg_path,
                render_path,
                comparison_path,
                scenario["title"],
            )
        )
        extra["comparison_path"] = str(comparison_path)
        parse_ok = extra.get("svg_saved", False)
    else:
        parse_ok = True
        save_text_overlay(image_path, result.text, overlay_path, scenario["title"])

    return {
        "id": scenario["id"],
        "image": str(image_path),
        "output": str(raw_path),
        "overlay": str(overlay_path),
        "kind": scenario["kind"],
        "temperature": scenario["temperature"],
        "top_p": scenario["top_p"],
        "max_tokens": MAX_TOKENS,
        "prompt_tokens": result.prompt_tokens,
        "generation_tokens": result.generation_tokens,
        "capped": result.generation_tokens >= MAX_TOKENS,
        "parse_ok": parse_ok,
        **extra,
    }

results = []
result_map = {}

def record_result(item):
    results[:] = [existing for existing in results if existing["id"] != item["id"]]
    results.append(item)
    result_map[item["id"]] = item
    return item

def display_result(item):
    lines = [
        f"### `{item['id']}`",
        "",
        f"- kind: `{item['kind']}`",
        f"- image: `{Path(item['image']).name}`",
        f"- output: `{Path(item['output']).name}`",
        f"- overlay: `{Path(item['overlay']).name}`",
        f"- generation tokens: `{item['generation_tokens']}`",
        f"- capped: `{item['capped']}`",
        f"- parse ok: `{item.get('parse_ok')}`",
    ]
    if item.get("alias_of"):
        lines.append(f"- alias of: `{item['alias_of']}`")
    if item["kind"] == "svg" and item.get("comparison_path"):
        lines.append(f"- comparison: `{Path(item['comparison_path']).name}`")
    display(Markdown("\n".join(lines)))
    if item["kind"] == "svg" and item.get("comparison_path"):
        display(Image.open(item["comparison_path"]))
    display(Image.open(item["overlay"]))
    return item

def run_and_show(scenario_id: str):
    item = record_result(run_scenario(SCENARIO_INDEX[scenario_id]))
    return display_result(item)

def materialize_alias(alias: str, target: str):
    if alias in result_map:
        return display_result(result_map[alias])
    if target not in result_map:
        raise KeyError(f"Run {target} before materializing alias {alias}")
    source = result_map[target]
    alias_output = OUTPUTS_DIR / f"{alias}.txt"
    alias_overlay = OVERLAYS_DIR / f"{alias}.png"
    shutil.copyfile(source["output"], alias_output)
    shutil.copyfile(source["overlay"], alias_overlay)
    alias_item = dict(source)
    alias_item["id"] = alias
    alias_item["output"] = str(alias_output)
    alias_item["overlay"] = str(alias_overlay)
    alias_item["alias_of"] = target
    record_result(alias_item)
    return display_result(alias_item)


## Example 1: Document Parsing

Runs the full layout extraction prompt on `demo_image1.jpg` and saves the structured output plus overlay.


In [ ]:
demo_document_parsing = run_and_show("demo_document_parsing")


## Example 2: Web Parsing

Runs the webpage layout parsing prompt on `webpage_1.png`.


In [ ]:
demo_web_parsing = run_and_show("demo_web_parsing")


## Example 3: Scene Spotting

Detects and labels text instances in the street-sign scene image.


In [ ]:
demo_scene_spotting = run_and_show("demo_scene_spotting")


## Example 4: Image To SVG

Generates SVG from `demo_image2.png`, then renders a side-by-side comparison before showing the final overlay.


In [ ]:
demo_svg = run_and_show("demo_svg")


## Example 5: General Image QA

Uses the general visual question-answering prompt on `demo_image3.jpg`.


In [ ]:
demo_general = run_and_show("demo_general")


## Example 6: HF Layout Alias

The upstream README includes `demo_hf_layout` as a separate example, but it reuses the same image and prompt as `demo_document_parsing`.
This section materializes the alias artifacts and displays that saved overlay here.


In [ ]:
demo_hf_layout = materialize_alias("demo_hf_layout", "demo_document_parsing")


## Example 7: Parser Image Default Alias

`parser_image_default` also reuses the same image and prompt as `demo_document_parsing`, so the notebook copies the saved artifacts into that alias name.


In [ ]:
parser_image_default = materialize_alias("parser_image_default", "demo_document_parsing")


## Example 8: Parser PDF Default

Runs the full layout prompt on the rendered first page of `demo_pdf1.pdf`.


In [ ]:
parser_pdf_default = run_and_show("parser_pdf_default")


## Example 9: Parser Layout Only

Extracts layout categories and boxes without the text payload.


In [ ]:
parser_layout_only = run_and_show("parser_layout_only")


## Example 10: Parser OCR

Runs plain OCR on the document image and saves the text overlay preview.


In [ ]:
parser_ocr = run_and_show("parser_ocr")


## Summary And Files

After all example sections run, this cell writes the summary files and contact sheet under `OUTPUT_ROOT`.


In [ ]:
missing = [key for key in SCENARIO_ORDER if key not in result_map]
if missing:
    raise RuntimeError(f"Run all example sections before the summary cell: {missing}")

ordered_results = [result_map[key] for key in SCENARIO_ORDER]

summary_lines = [
    "# dots.mocr README examples via MLX-VLM",
    "",
    f"Output root: {OUTPUT_ROOT}",
    "",
    "| example | kind | tokens | capped | parse_ok | overlay | output |",
    "| --- | --- | ---: | --- | --- | --- | --- |",
]
for item in ordered_results:
    summary_lines.append(
        f"| {item['id']} | {item['kind']} | {item['generation_tokens']} | {item['capped']} | {item.get('parse_ok')} | {Path(item['overlay']).name} | {Path(item['output']).name} |"
    )
summary_md = "\n".join(summary_lines) + "\n"

(OUTPUT_ROOT / "results.json").write_text(
    json.dumps(ordered_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUTPUT_ROOT / "summary.md").write_text(summary_md, encoding="utf-8")
make_contact_sheet(ordered_results, OUTPUT_ROOT / "overlay_contact_sheet.png")

display(Markdown(summary_md))
display(Image.open(OUTPUT_ROOT / "overlay_contact_sheet.png"))
print(OUTPUT_ROOT)


Useful files under `OUTPUT_ROOT`:
- `results.json`
- `summary.md`
- `overlay_contact_sheet.png`
- `outputs/*.txt`
- `overlays/*.png`
- `rendered/demo_svg_comparison.png`
